In [ ]:
import os

import numpy as np
from robokit.datasets.libero.libero_force import LiberoH5FrameDataset, MergeDataset
from robokit.debug_utils.printer import print_batch
from robokit.debug_utils.images import concatenate_rgb_images, plot_action_wrt_time, plot_force_sensor_wrt_time, save_frames_as_video

In [5]:
dataset_root = "/home/geyuan/code/LIBERO-FT/libero/datasets/"
dataset_subname = "libero_90"
hdf5_fn = "KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer_demo.hdf5"
hdf5_path = os.path.join(dataset_root, dataset_subname, hdf5_fn)
hdf5_save_path = "tmp_replayed_wrench.hdf5"

single_dataset = LiberoH5FrameDataset(
    hdf5_save_path,
)
print("len(single_dataset):", len(single_dataset))
print_batch("LiberoH5FrameDataset[0]", single_dataset[0])

print("action:", single_dataset[100]["actions"])

len(single_dataset): 11723
LiberoH5FrameDataset[0]: Dict, keys=['demo_key', 't', 'obs', 'text_instruction', 'actions', 'dones', 'rewards', 'robot_states', 'states', 'replay_reset_every']
--demo_key: <class 'str'>, len=6, value='demo_0'
--t: <class 'int'>, value=0
--obs: Dict, keys=['agentview_rgb', 'ee_ori', 'ee_pos', 'ee_states', 'eye_in_hand_rgb', 'gripper_states', 'joint_states', 'wrenches']
----agentview_rgb, <class 'numpy.ndarray'>, shape=(128, 128, 3), min=0.0000, max=255.0000, dtype=uint8
----ee_ori, <class 'numpy.ndarray'>, shape=(3,), min=-0.0642, max=3.1760, dtype=float64
----ee_pos, <class 'numpy.ndarray'>, shape=(3,), min=-0.2018, max=1.1802, dtype=float64
----ee_states, <class 'numpy.ndarray'>, shape=(6,), min=-0.2018, max=3.1760, dtype=float64
----eye_in_hand_rgb, <class 'numpy.ndarray'>, shape=(128, 128, 3), min=0.0000, max=237.0000, dtype=uint8
----gripper_states, <class 'numpy.ndarray'>, shape=(2,), min=-0.0340, max=0.0341, dtype=float64
----joint_states, <class 'numpy

In [10]:
n_obs = 20
ds2 = MergeDataset(
    [single_dataset, single_dataset, single_dataset],
    n_obs=n_obs,
    chunk_size=140,
    obs_keys=["agentview_rgb", "eye_in_hand_rgb", "ee_states", "gripper_states","wrenches"]
)
# print("len(ds2):", len(ds2))
# print_batch("MergeDataset[0]", ds2[0])
# '''
# MergeDataset[0]: Dict, keys=['demo_key', 't', 'text_instruction', 'history', 'future', 'action']
# --demo_key: <class 'str'>, len=6, value='demo_0'
# --t: <class 'int'>, value=0
# --text_instruction: <class 'str'>, len=77, value='KITCHEN SCENE4 close the bottom drawer of the cabinet and open the top drawer'
# --history: Dict, keys=['obs', 'state']
# ----obs: Dict, keys=['agentview_rgb', 'eye_in_hand_rgb', 'ee_states', 'gripper_states', 'wrenches']
# ------agentview_rgb, <class 'numpy.ndarray'>, shape=(2, 128, 128, 3), min=0.0000, max=255.0000, dtype=uint8
# ------eye_in_hand_rgb, <class 'numpy.ndarray'>, shape=(2, 128, 128, 3), min=0.0000, max=237.0000, dtype=uint8
# ------ee_states, <class 'numpy.ndarray'>, shape=(2, 6), min=-0.2018, max=3.1760, dtype=float64
# ------gripper_states, <class 'numpy.ndarray'>, shape=(2, 2), min=-0.0340, max=0.0341, dtype=float64
# ------wrenches, <class 'numpy.ndarray'>, shape=(2, 6), min=-3.3128, max=1.0444, dtype=float32
# ----state: Dict, keys=['states', 'robot_states']
# ------states, <class 'numpy.ndarray'>, shape=(2, 51), min=-2.4278, max=2.2271, dtype=float64
# ------robot_states, <class 'numpy.ndarray'>, shape=(2, 9), min=-0.2018, max=1.1802, dtype=float64
# --future: Dict, keys=['obs', 'state']
# ----obs: Dict, keys=['agentview_rgb', 'eye_in_hand_rgb', 'ee_states', 'gripper_states', 'wrenches']
# ------agentview_rgb, <class 'numpy.ndarray'>, shape=(32, 128, 128, 3), min=0.0000, max=255.0000, dtype=uint8
# ------eye_in_hand_rgb, <class 'numpy.ndarray'>, shape=(32, 128, 128, 3), min=0.0000, max=238.0000, dtype=uint8
# ------ee_states, <class 'numpy.ndarray'>, shape=(32, 6), min=-0.2036, max=3.3343, dtype=float64
# ------gripper_states, <class 'numpy.ndarray'>, shape=(32, 2), min=-0.0395, max=0.0396, dtype=float64
# ------wrenches, <class 'numpy.ndarray'>, shape=(32, 6), min=-5.2036, max=1.1171, dtype=float32
# ----state: Dict, keys=['states', 'robot_states']
# ------states, <class 'numpy.ndarray'>, shape=(32, 51), min=-2.4306, max=2.2255, dtype=float64
# ------robot_states, <class 'numpy.ndarray'>, shape=(32, 9), min=-0.2036, max=1.2024, dtype=float64
# --action, <class 'numpy.ndarray'>, shape=(32, 7), min=-1.0000, max=0.9375, dtype=float64
# '''

stats = ds2.compute_statistics(
    keys={
        "obs": ["wrenches", "ee_states"],
        "state": [],
        "action": ["actions"],
    }
)
print_batch("Libero Dataset Statistics", stats)

{
  "obs.wrenches": {
    "count": 35169,
    "mean": [1.843, 3.2048, 4.8427, -0.1094, 0.4145, -0.2352],
    "std": [13.7607, 16.7166, 23.3068, 3.2562, 3.3553, 1.5179],
    "min": [-129.3165, -408.8553, -111.3397, -18.2925, -36.0573, -28.0831],
    "max": [358.5761, 251.0575, 444.5059, 209.1904, 139.1617, 27.7514],
    "p01": [-32.6021, -23.7529, -16.8016, -5.5376, -7.3938, -5.8372],
    "p99": [49.6821, 52.7416, 91.8689, 3.2273, 11.7109, 3.3115]
  },
  "obs.ee_states": {
    "count": 35169,
    "mean": [-0.0054, 0.0946, 1.0694, 2.7379, -2.1385, -0.786],
    "std": [0.0624, 0.0821, 0.0811, 0.284, 0.7517, 0.4193],
    "min": [-0.219, -0.0989, 0.9104, 1.785, -3.1398, -2.0896],
    "max": [0.135, 0.2382, 1.2396, 3.5025, 0.1154, 0.1965],
    "p01": [-0.1991, -0.066, 0.9154, 2.1359, -2.9069, -1.8728],
    "p99": [0.0942, 0.2172, 1.1922, 3.3524, 0.0024, -0.0357]
  },
  "action.actions": {
    "count": 35169,
    "mean": [0.0083, -0.0014, -0.1489, 0.0243, 0.0099, -0.0561, -1.0],
    "std": [0

In [4]:
index = 0

history_agentview_rgb = ds2[index]["history"]["obs"]["agentview_rgb"]  # (n_obs, H, W, 3)
history_eyeinhand_rgb = ds2[index]["history"]["obs"]["eye_in_hand_rgb"]  # (n_obs, H, W, 3)
history_wrenches = ds2[index]["history"]["obs"]["wrenches"]  # (n_obs, 6)
future_agentview_rgb = ds2[index]["future"]["obs"]["agentview_rgb"]  # (chunk_size, H, W, 3)
future_eyeinhand_rgb = ds2[index]["future"]["obs"]["eye_in_hand_rgb"]  # (chunk_size, H, W, 3)
future_wrenches = ds2[index]["future"]["obs"]["wrenches"]  # (chunk_size, 6)
action = ds2[index]["action"]  # (chunk_size, action_dim)

agent_rgbs = np.concatenate((history_agentview_rgb[n_obs - 1:], future_agentview_rgb), axis=0)
eyeinhand_rgbs = np.concatenate((history_eyeinhand_rgb[n_obs - 1:], future_eyeinhand_rgb), axis=0)
T = agent_rgbs.shape[0]
vis_agent_images = [
    concatenate_rgb_images(agent_rgb, eyeinhand_rgb, vertical=True, resize_ratio=2.0) for agent_rgb, eyeinhand_rgb in zip(agent_rgbs, eyeinhand_rgbs)
]
action_frames, *_ = plot_action_wrt_time(action)
force_frames, *_ = plot_force_sensor_wrt_time(future_wrenches)

vis_merged_frames = [
    concatenate_rgb_images(camera_frame, action_frame, resize_ratio=1.0) for camera_frame, action_frame in zip(vis_agent_images, action_frames)
]
vis_merged_frames = [
    concatenate_rgb_images(merged_frame, force_frame, resize_ratio=1.0)
    for merged_frame, force_frame in zip(vis_merged_frames, force_frames)
]

save_frames_as_video(
    vis_merged_frames,
    "tmp_libero_replayed_wrench.mp4",
    fps=10,
)


NameError: name 'agent_rgb' is not defined

In [9]:
import h5py, numpy as np

h5_path = "/home/geyuan/code/LIBERO-FT/libero/datasets/libero_90/KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer_demo.hdf5"

with h5py.File(h5_path, "r") as f:
    dk = sorted(list(f["data"].keys()), key=lambda x: int(x.split("_")[1]))[20]
    g = f["data"][dk]
    a = g["actions"][...]  # (T, 7) usually
    print("actions shape:", a.shape, "dtype:", a.dtype)
    print("last-dim min/max:", a[:, -1].min(), a[:, -1].max())
    print("unique(last-dim) sample:", np.unique(a[:, -1])[:10], " (#unique:", np.unique(a[:, -1]).size, ")")


actions shape: (244, 7) dtype: float64
last-dim min/max: -1.0 -1.0
unique(last-dim) sample: [-1.]  (#unique: 1 )
